In [2]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import ByteLevel
from tokenizers.normalizers import Sequence, NFD, StripAccents
from tokenizers.processors import ByteLevel as ByteLevelProcessor
import os

In [3]:
import numpy as np
from tqdm import tqdm

In [4]:
import torch
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

In [5]:
import torch.nn as nn

In [ ]:
# PATHS
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

TRAIN_FILE = os.path.join(BASE_DIR, "Data", "dataset", "train.txt")
SAVE_PATH = os.path.join(BASE_DIR, "tokenizer", "code_tokenizer.json")

# CONFIG
VOCAB_SIZE = 50000
MIN_FREQ = 2

# INIT TOKENIZER
tokenizer = Tokenizer(BPE(unk_token="[UNK]"))

tokenizer.normalizer = Sequence([
    NFD(),
    StripAccents()
])

tokenizer.pre_tokenizer = ByteLevel(add_prefix_space=True)

# TRAINER
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,
    min_frequency=MIN_FREQ,
    special_tokens=[
        "[PAD]",
        "[UNK]",
        "[CLS]",
        "[SEP]",
        "[MASK]"
    ]
)

# TRAIN
print("Training tokenizer on train.txt ...")
tokenizer.train([TRAIN_FILE], trainer)

# POST PROCESSING
tokenizer.post_processor = ByteLevelProcessor()

# SAVE
os.makedirs(os.path.dirname(SAVE_PATH), exist_ok=True)
tokenizer.save(SAVE_PATH)

print(f"Tokenizer saved at: {SAVE_PATH}")

Training tokenizer on train.txt ...
Tokenizer saved at: d:\Skills\Machine_Learning\GPT\tokenizer\code_tokenizer.json


In [6]:
# PATHS
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

DATASET_DIR = os.path.join(BASE_DIR, "Data", "dataset")
OUTPUT_DIR = os.path.join(BASE_DIR, "Data", "tokenized")
TOKENIZER_PATH = os.path.join(BASE_DIR, "tokenizer", "code_tokenizer.json")

# LOAD TOKENIZER
tokenizer = Tokenizer.from_file(TOKENIZER_PATH)

# FUNCTION: STREAM TOKENIZE
def tokenize_file(input_path, output_path):
    print(f"\nProcessing: {input_path}")

    all_ids = []

    with open(input_path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc="Tokenizing"):
            ids = tokenizer.encode(line).ids
            all_ids.extend(ids)

    
    # SAVE
    print("Saving...")

    dtype = np.uint16 if max(all_ids) < 65536 else np.uint32
    arr = np.array(all_ids, dtype=dtype)

    arr.tofile(output_path)

    print(f"Saved: {output_path}")
    print(f"Total tokens: {len(arr)}")


# RUN FOR ALL SPLITS
splits = ["train", "val", "test"]

for split in splits:
    input_path = os.path.join(DATASET_DIR, f"{split}.txt")
    output_path = os.path.join(OUTPUT_DIR, f"{split}.bin")

    tokenize_file(input_path, output_path)


Processing: d:\Skills\Machine_Learning\GPT\Data\dataset\train.txt


Tokenizing: 448494it [00:15, 28301.85it/s]


Saving...
Saved: d:\Skills\Machine_Learning\GPT\Data\tokenized\train.bin
Total tokens: 4731363

Processing: d:\Skills\Machine_Learning\GPT\Data\dataset\val.txt


Tokenizing: 55325it [00:02, 18638.16it/s]


Saving...
Saved: d:\Skills\Machine_Learning\GPT\Data\tokenized\val.bin
Total tokens: 594161

Processing: d:\Skills\Machine_Learning\GPT\Data\dataset\test.txt


Tokenizing: 55601it [00:02, 23403.60it/s]


Saving...
Saved: d:\Skills\Machine_Learning\GPT\Data\tokenized\test.bin
Total tokens: 595799


In [7]:
class CodeDataset(Dataset):
    def __init__(self, bin_file, block_size=1024):
        self.block_size = block_size
        
        # Load tokenized data
        self.data = np.fromfile(bin_file, dtype=np.uint16)
        
        # Total number of training samples
        self.n_samples = len(self.data) - block_size

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        # Input sequence
        x = self.data[idx : idx + self.block_size]
        
        # Target sequence (shifted by 1)
        y = self.data[idx + 1 : idx + self.block_size + 1]
        
        return (
            torch.tensor(x, dtype=torch.long),
            torch.tensor(y, dtype=torch.long)
        )

In [8]:
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
TRAIN_BIN = os.path.join(BASE_DIR, "Data", "tokenized", "train.bin")

train_dataset = CodeDataset(TRAIN_BIN, block_size=1024)

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

In [9]:
class GPTEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, block_size):
        super().__init__()
        
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.position_embedding = nn.Embedding(block_size, embed_dim)

    def forward(self, x):
        B, T = x.shape  # Batch, Time
        
        token_emb = self.token_embedding(x)  # (B, T, C)
        
        positions = torch.arange(T, device=x.device)  # (T,)
        pos_emb = self.position_embedding(positions)  # (T, C)
        
        return token_emb + pos_emb

In [10]:
class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.qkv = nn.Linear(embed_dim, 3 * embed_dim)
        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, T, C = x.shape
        
        qkv = self.qkv(x)  # (B, T, 3C)
        q, k, v = qkv.chunk(3, dim=-1)
        
        # reshape for multi-head
        q = q.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.num_heads, self.head_dim).transpose(1, 2)

        # attention scores
        att = (q @ k.transpose(-2, -1)) / (self.head_dim ** 0.5)
        
        # causal mask
        mask = torch.tril(torch.ones(T, T, device=x.device))
        att = att.masked_fill(mask == 0, float('-inf'))
        
        att = torch.softmax(att, dim=-1)
        
        out = att @ v  # (B, heads, T, head_dim)
        
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        
        return self.out(out)

In [11]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim):
        super().__init__()
        
        self.net = nn.Sequential(
            nn.Linear(embed_dim, 4 * embed_dim),
            nn.GELU(),
            nn.Linear(4 * embed_dim, embed_dim),
        )

    def forward(self, x):
        return self.net(x)

In [12]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super().__init__()
        
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = CausalSelfAttention(embed_dim, num_heads)
        
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ff = FeedForward(embed_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

In [13]:
class GPT(nn.Module):
    def __init__(self, vocab_size, embed_dim=512, block_size=1024, num_heads=8, num_layers=6):
        super().__init__()
        
        self.embedding = GPTEmbedding(vocab_size, embed_dim, block_size)
        
        self.blocks = nn.Sequential(*[
            TransformerBlock(embed_dim, num_heads)
            for _ in range(num_layers)
        ])
        
        self.ln_f = nn.LayerNorm(embed_dim)
        
        self.head = nn.Linear(embed_dim, vocab_size)

    def forward(self, x, targets=None):
        x = self.embedding(x)
        x = self.blocks(x)
        x = self.ln_f(x)
        
        logits = self.head(x)  # (B, T, vocab_size)
        
        loss = None
        if targets is not None:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            
            loss = nn.CrossEntropyLoss()(logits, targets)
        
        return logits, loss

In [15]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = GPT(vocab_size=50000).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

In [16]:
def train_step(model, batch, optimizer, device):
    model.train()
    
    x, y = batch
    x = x.to(device)
    y = y.to(device)
    
    # Forward pass
    logits, loss = model(x, y)
    
    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    
    # Optimizer step
    optimizer.step()
    
    return loss.item()

In [ ]:
max_steps = 15000  

step_count = 0
total_loss = 0

for epoch in range(1000):  # dummy loop (won't actually reach 1000)
    for step, batch in enumerate(train_loader):
        
        loss = train_step(model, batch, optimizer, device)
        
        total_loss += loss
        
        if step_count % 100 == 0:
            avg_loss = total_loss / (step_count + 1)
            print(f"Step {step_count} | Loss: {loss:.4f} | Avg Loss: {avg_loss:.4f}")
        
        step_count += 1
        
        # Stop condition
        if step_count >= max_steps:
            break
    
    if step_count >= max_steps:
        break

c:\Users\Infinity\anaconda3\envs\ts_env\lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
